# Assignment 09 — Multi-Head Latent Attention (100 points)

This assignment mirrors **USAAIO 2025 Round 2 Problem 2, Parts 9-11**. You will implement MLA, prove GQA $\subseteq$ MLA via SVD, convert GQA weights to MLA form, and construct a counterexample showing GQA $\subsetneq$ MLA.

**Notation:**
- $B$: batch size, $L$: sequence length, $D$: model dimension
- $H$: number of heads, $D_{qk}$: query/key dim, $D_v$: value dim
- $r$: MLA latent rank ($r \ll D$)
- $W^{DKV} \in \mathbb{R}^{D \times r}$: shared down-projection
- $W^{UK}_h \in \mathbb{R}^{r \times D_{qk}}$: per-head key up-projection
- $W^{UV}_h \in \mathbb{R}^{r \times D_v}$: per-head value up-projection
- MLA key: $K_h = X W^{DKV} W^{UK}_h$ so $W^K_h = W^{DKV} W^{UK}_h \in \mathbb{R}^{D \times D_{qk}}$

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

**WARNING**: You may only use `torch`, `torch.nn`, `torch.nn.functional`, and `numpy`. No other imports are allowed.

## Part 1 (10 points, non-coding task)

**MLA decomposition shapes.**

Given $D = 512, H = 8, D_{qk} = 64, D_v = 64, r = 32$:

1. Shape of $W^{DKV}$. (2 points)
2. Shape of $W^{UK}_h$ for one head. (2 points)
3. Shape of $W^K_h = W^{DKV} \cdot W^{UK}_h$. Verify dimensions. (2 points)
4. Shape of the compressed representation $C = X W^{DKV}$ for $X \in \mathbb{R}^{B \times L \times D}$. (2 points)
5. What is the maximum rank of $W^K_h$? (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 2 (15 points, coding task)

Implement `MyMLA` as an `nn.Module`.

Constructor: `MyMLA(D, H, D_qk, D_v, r)`
- `W_Q`: `nn.Linear(D, H * D_qk, bias=False)` — standard query projection
- `W_DKV`: `nn.Linear(D, r, bias=False)` — shared down-projection
- `W_UK`: `nn.Linear(r, H * D_qk, bias=False)` — key up-projection (all heads)
- `W_UV`: `nn.Linear(r, H * D_v, bias=False)` — value up-projection (all heads)
- `W_O`: `nn.Linear(H * D_v, D, bias=False)` — output projection

Forward: `forward(X)` where $X: (B, L, D)$, returns $(B, L, D)$.

Steps:
1. $Q = X W^Q$ — standard query projection
2. $C = X W^{DKV}$ — compress input to latent space
3. $K = C W^{UK}$, $V = C W^{UV}$ — up-project to K, V
4. Standard attention with reshape/permute (NO LOOPS)

Shape annotations required.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 3 (5 points, coding task)

Test `MyMLA`.

1. Create with $D=64, H=4, D_{qk}=16, D_v=16, r=8$.
2. Random input $(2, 10, 64)$.
3. Assert output shape $(2, 10, 64)$.
4. Count parameters. Compare with MHA having the same $D, H, D_{qk}, D_v$.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

---

We will now explore the relationship between GQA and MLA. Recall that in GQA with $G$ groups, the effective key matrix for a head in group $g$ is $W^K_g \in \mathbb{R}^{D \times D_{qk}}$, shared across $H/G$ query heads.

---

## Part 4 (15 points, non-coding task)

**Proof: GQA $\subseteq$ MLA (via SVD)**

Prove that any GQA configuration can be represented as an MLA configuration.

Your proof should:
1. Start with GQA having $G$ groups with weight matrices $W^K_1, \ldots, W^K_G$ and $W^V_1, \ldots, W^V_G$. (2 points)
2. Stack them: $W_{\text{all}} = [W^K_1 | \cdots | W^K_G | W^V_1 | \cdots | W^V_G]$. (2 points)
3. Take the SVD: $W_{\text{all}} = U \Sigma V^T$. (3 points)
4. Define $W^{DKV} = U \Sigma$ (or $U[:, :r] \Sigma[:r, :r]$ for a rank-$r$ truncation). (3 points)
5. Show how to extract $W^{UK}_h$ and $W^{UV}_h$ from $V^T$ such that $W^{DKV} W^{UK}_h = W^K_g$ for heads in group $g$. (5 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 5 (10 points, coding task)

**GQA-to-MLA conversion with NumPy.**

Implement `gqa_to_mla(W_K_groups, W_V_groups)` that:
1. Takes lists of $G$ key matrices (each $(D, D_{qk})$) and $G$ value matrices (each $(D, D_v)$) as NumPy arrays.
2. Stacks them horizontally.
3. Computes SVD.
4. Returns $W^{DKV}$, list of $W^{UK}_g$, list of $W^{UV}_g$.

Verify:
- Create random GQA weights with $D=16, D_{qk}=4, D_v=4, G=2$.
- Convert to MLA.
- Assert $W^{DKV} @ W^{UK}_g \approx W^K_g$ for each group.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 6 (15 points, non-coding task)

**Proof: GQA $\subsetneq$ MLA (counterexample)**

Prove that MLA is strictly more expressive than GQA. That is, there exists an MLA configuration that no GQA can represent.

Construct a specific counterexample:
1. Choose $H = 4, D = 4, D_{qk} = 2, r = 4$. (2 points)
2. Define $W^{DKV} = I_4$ (identity) and choose 4 DISTINCT $W^{UK}_h$ so that all $W^K_h = W^{DKV} W^{UK}_h$ are distinct. (4 points)
3. Show that for GQA with $G = 1$: all heads share one $W^K$, so cannot have distinct key matrices. Contradiction. (3 points)
4. Show that for GQA with $G = 2$: at most 2 distinct key matrices, but we have 4. Contradiction. (3 points)
5. The only GQA option is $G = 4 = H$ (MHA), but its KV-cache is $2 \times H \times D_{qk} = 16 > r = 4$. Conclude GQA $\subsetneq$ MLA. (3 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 7 (5 points, coding task)

Verify the counterexample numerically.

1. Create $W^{DKV} = I_4$ and four distinct $W^{UK}_h$ matrices (each $4 \times 2$).
2. Compute $W^K_h = W^{DKV} @ W^{UK}_h$ for each head.
3. Assert all four $W^K_h$ are distinct (use `np.allclose` to check pairwise inequality).
4. For any GQA with $G = 2$: heads 1-2 share one $W^K$, heads 3-4 share another. Show this forces $W^K_1 = W^K_2$ and $W^K_3 = W^K_4$, which contradicts distinctness.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 8 (5 points, non-coding task)

**Inclusion hierarchy summary.**

1. State the complete inclusion chain: MHA $\subseteq$ GQA $\subseteq$ MLA. (1 point)
2. For each inclusion, state why it is $\subseteq$ (which parameter setting). (2 points)
3. State which inclusions are strict ($\subsetneq$) and cite the proof/counterexample. (2 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 9 (10 points, coding task)

Verify that MLA with appropriate weights produces the same output as MHA.

1. Create an MHA module with $D=16, H=2, D_{qk}=4, D_v=4$.
2. Extract the key weights for each head: $W^K_h \in \mathbb{R}^{16 \times 4}$.
3. Stack and SVD to get $W^{DKV}$ and $W^{UK}_h$.
4. Create an MLA module with the appropriate $r$.
5. Set the MLA weights to match the decomposition.
6. Pass the same input through both and assert the outputs match.

Reasoning is not required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

## Part 10 (10 points, non-coding task)

**When is MLA beneficial over GQA?**

1. For MLA with rank $r$ and GQA with $G$ groups, when is MLA's KV-cache smaller? Express as an inequality involving $r$, $G$, and $D_{qk}$. (3 points)
2. For $D = 8192, H = 128, D_{qk} = 128$: compute the KV-cache per position for GQA with $G = 1$ (MQA) and MLA with $r = 512$. Which is better? (3 points)
3. MLA can have all $H$ heads with distinct key matrices while caching only $r$ values. GQA with $G$ groups can have at most $G$ distinct key matrices. Explain why this matters for model quality. (4 points)

Reasoning is required.

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """